# A chart over time and a time-lapse (Python)

**Fullstack Remote Sensing** · code by **Rifky Nauval Hendrawan, Remote Sensing Specialist** · [Chapter: Composites and Visualisation](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/en/13-visualisation.html) · [Bab (Bahasa Indonesia)](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/id/13-visualisation.html) · [Open the JavaScript in the Code Editor](https://code.earthengine.google.com/?scriptPath=users/rifkynauvalhsp/Fullstack-Remote-Sensing:ch13_charts_timelapse)

Run the cells from top to bottom. The first cell installs what Colab lacks, fetches the book's data and helpers, and signs you in to Earth Engine with your own Google Cloud project ([how to get one](https://developers.google.com/earth-engine/guides/access)).

*Jalankan sel dari atas ke bawah. Sel pertama memasang paket, mengambil data buku, dan masuk ke Earth Engine dengan proyek Google Cloud Anda sendiri.*

In [ ]:
# @title Setup: packages, book data, Earth Engine sign-in
!pip -q install earthengine-api geemap tabulate
!test -d /content/book || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rifkynhsp-byte/Fullstack-Remote-Sensing.git /content/book
!git -C /content/book sparse-checkout set tools scripts data   # only what the code needs, not the website
import sys, os; sys.path[:0] = ['/content/book/tools', '/content/book/scripts/py']; os.chdir('/content/book')
__file__ = '/content/book/scripts/py/ch13_charts_timelapse.py'   # the script finds the book's data relative to itself
PROJECT = "your-cloud-project"  # @param {type:"string"}
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
_init = ee.Initialize; ee.Initialize = lambda *a, **k: None   # the script must not sign in again

CHAPTER 13 | NDVI through time at one Bandung point, and an animated GIF of
the IKN core zone, 2019 to 2024, in Python.

In [ ]:
# Fullstack Remote Sensing | Code by Rifky Nauval Hendrawan, Remote Sensing Specialist
# MIT licence: free to use and adapt; please keep this credit line.

"""
CHAPTER 13 | NDVI through time at one Bandung point, and an animated GIF of
the IKN core zone, 2019 to 2024, in Python.
"""

import ee
import matplotlib.pyplot as plt
import pandas as pd

point = ee.Geometry.Point([107.6262, -6.9230])   # Bandung

In [ ]:
def add_ndvi(image):
    qa = image.select("QA_PIXEL")
    clear = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    sr = image.select(["SR_B4", "SR_B5"]).multiply(0.0000275).add(-0.2)
    return (sr.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
            .updateMask(clear).copyProperties(image, ["system:time_start"]))


ndvi_series = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
               .filterBounds(point).filterDate("2013-04-01", "2025-01-01")
               .map(add_ndvi))

# ui.Chart.image.series in Python: sample every image at the point
series = ee.FeatureCollection(ndvi_series.map(lambda im: ee.Feature(None, {
    "date": im.date().format("YYYY-MM-dd"),
    "NDVI": im.reduceRegion(ee.Reducer.first(), point, 30).get("NDVI"),
}))).filter(ee.Filter.notNull(["NDVI"]))

In [ ]:
def plot_ndvi_series(df):
    """Every clear observation, plus a 90-day rolling median to show the rhythm."""
    df = df.assign(date=pd.to_datetime(df["date"])).sort_values("date")
    smooth = df.set_index("date")["NDVI"].rolling("90D").median()
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.scatter(df["date"], df["NDVI"], s=8, color="#74A901", alpha=0.6,
               label="clear observations")
    ax.plot(smooth.index, smooth.values, color="#056201", lw=1.5, label="90-day median")
    ax.set_ylabel("NDVI")
    ax.set_title("NDVI over time, one Bandung pixel", loc="left")
    ax.legend(frameon=False, fontsize=8)
    return fig


# The time-lapse
ikn = ee.Geometry.Rectangle([116.66, -1.01, 116.75, -0.93])

In [ ]:
def yearly_frame(year):
    def mask(image):
        scl = image.select("SCL")
        return image.updateMask(scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)))
    composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                 .filterBounds(ikn)
                 .filterDate(ee.Date.fromYMD(year, 5, 1), ee.Date.fromYMD(year, 10, 31))
                 .map(mask).median())
    return (composite.visualize(bands=["B4", "B3", "B2"], min=200, max=2200, gamma=1.2)
            .set("year", year))


YEARS = list(range(2019, 2025))
frame_list = [yearly_frame(y) for y in YEARS]
frames = ee.ImageCollection(frame_list)

In [ ]:
# frames.getVideoThumbURL(...) is the one-line route, and over this area it
# fails with "User memory limit exceeded": six medians are computed in one
# request. Asking for each frame separately and stitching the GIF in Python
# stays under the limit. That trade, one big request against many small
# ones, comes up again in chapter “Statistics and Modelling You Actually Need”.


def products():
    return [
        {"kind": "chart", "name": "ch13-ndvi-series", "data": series,
         "plot": plot_ndvi_series,
         "caption": "Every clear Landsat 8 observation at one Bandung pixel since 2013. "
                    "The gaps are cloud; the rolling median is the pattern a reader "
                    "should take away."},
        {"kind": "animation", "name": "ch13-ikn-timelapse", "frames": frame_list,
         "region": ikn, "labels": [f"IKN core zone, {y}" for y in YEARS], "fps": 1,
         "caption": "One cloud-masked Sentinel-2 composite per year (May to October), "
                    "2019 to 2024: the Nusantara capital core zone in East Kalimantan. "
                    "The bare patches in 2019 are harvested plantation blocks, not "
                    "construction; the road grid and core buildings appear from 2022."},
        {"kind": "map", "name": "ch13-ikn-2024", "image": frame_list[-1],
         "vis": {"bands": ["vis-red", "vis-green", "vis-blue"], "min": 0, "max": 255},
         "region": ikn, "title": "IKN core zone, 2024",
         "source": "Sentinel-2 SR Harmonized, May to October 2024. GEE.",
         "caption": "The last frame as a map, with coordinates and a scale bar."},
    ]

### The chapter's maps, charts and tables
The same products the book shows, drawn here. Maps also open as an interactive layer.

In [ ]:
from colab_render import show_all
show_all(products())